it extreact the required items first then will do the removal keyowrd search
- it first extract the description, androidmanifest path and readme then perform the search for removal keyword


In [ ]:
# === Step 3: Advanced Removal Keyword Detection ===
import os
import re
import requests
import pandas as pd
from base64 import b64decode
from dotenv import load_dotenv
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")
token_index = 0

def get_auth_header():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {"Authorization": f"token {token}"}

# ✅ Output folder path
OUTPUT_DIR = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Define keyword baskets ===
keywords = ['example', 'sample', 'demo', 'test', 'debug', 'presentation', 'module', 'components', 'lib', 'library',
            'sdk', 'utils', 'utility', 'plugin', 'widget', 'playground', 'framework', 'architecture', 'skeleton',
            'collection', 'starting point', 'protocol', 'benchmark', 'hackathon', 'classroom', 'course', 'exercise',
            'assignment', 'homework', 'assessment', 'interview', 'asset', 'template', 'catalog', 'tutorial', 'tool']

preceded_by = ['this', 'is a', 'is an', 'our', 'my']
not_preceded_by = ['using', 'with']

# === GitHub API fetch functions ===
def fetch_repo_metadata(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}"
    resp = requests.get(url, headers=get_auth_header())
    if resp.status_code == 200:
        return resp.json().get("description", "")
    return ""

def fetch_manifest_paths(repo_full_name):
    url = f"https://api.github.com/search/code?q=filename:AndroidManifest.xml+repo:{repo_full_name}"
    resp = requests.get(url, headers=get_auth_header())
    if resp.status_code == 200:
        return [item["path"] for item in resp.json().get("items", [])]
    return []

def fetch_readme_content(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}/readme"
    resp = requests.get(url, headers=get_auth_header())
    if resp.status_code == 200:
        content = resp.json().get("content", "")
        return b64decode(content).decode('utf-8', errors='ignore')
    return ""

# === Load input ===
input_path = os.path.join(OUTPUT_DIR, "step2_manifest_final_output.csv")
df = pd.read_csv(input_path)
df = df[df["Valid_Repo_Step2"] == "yes"].copy()
df["Repository"] = df["html_url"].apply(lambda url: '/'.join(url.strip('/').split('/')[-2:]))

# === Detection ===
removal_flags = []
for idx, row in df.iterrows():
    repo = row['Repository']
    html_url = row['html_url']
    print(f"[{idx + 1}/{len(df)}] Reviewing {repo}...")

    try:
        description = fetch_repo_metadata(repo)
        manifest_paths = fetch_manifest_paths(repo)
        readme = fetch_readme_content(repo)
        flagged = False

        if any(k in path.lower() for k in keywords for path in manifest_paths):
            flagged = True

        if any(k in repo.lower() for k in keywords):
            flagged = True

        if any(
            re.search(rf'(?<!\S){re.escape(k)}(?!\S)', str(description), re.IGNORECASE) and
            not any(re.search(rf'(?<!\S){re.escape(word)}\s+(\S+\s+){{0,4}}{re.escape(k)}(?!\S)', str(description), re.IGNORECASE)
                    for word in not_preceded_by)
            for k in keywords
        ):
            flagged = True

        if any(
            re.search(rf'(?<!\S){re.escape(k)}(?!\S)', readme, re.IGNORECASE) and
            any(re.search(rf'(?<!\S){re.escape(p)}\s+(\S+\s+){{0,4}}{re.escape(k)}(?!\S)', readme, re.IGNORECASE)
                for p in preceded_by)
            for k in keywords
        ):
            flagged = True

        removal_flags.append("no" if not flagged else "yes")

    except Exception as e:
        print(f"❌ Error processing {repo}: {e}")
        removal_flags.append("error")
        sleep(1)

# === Final assignment ===
df["Valid_Repo_Step3"] = ["yes" if flag == "no" else "no" for flag in removal_flags]
df["removal_keyword_flag"] = removal_flags

output_path = os.path.join(OUTPUT_DIR, "step3_removal_keyword_output.csv")
df.to_csv(output_path, index=False)
print(f"✅ Step 3 complete: {output_path} saved.")


[1/25812] Processing https://github.com/0015/ThatProject
[2/25812] Processing https://github.com/008chen/InterpolatorShow
[3/25812] Processing https://github.com/00ec454/Ask
[4/25812] Processing https://github.com/00ec454/pop
[5/25812] Processing https://github.com/00-Evan/shattered-pixel-dungeon
[6/25812] Processing https://github.com/06peng/FrescoDemo
[7/25812] Processing https://github.com/08carmelo/android-keeplive
[8/25812] Processing https://github.com/0maru/twitter_login
[9/25812] Processing https://github.com/0niel/university-app
[10/25812] Processing https://github.com/0ranko0P/AutoDark
[11/25812] Processing https://github.com/0x4f53/Wristkey
[12/25812] Processing https://github.com/0x5e/RubiksCubeSolver
[13/25812] Processing https://github.com/0x7c13/Pal3.Unity
[14/25812] Processing https://github.com/0xbad1d3a5/Kaku
[15/25812] Processing https://github.com/0xchat-app/0xchat-app-main
[16/25812] Processing https://github.com/0xchat-app/0xchat-core
[17/25812] Processing https:/